# Virality Research Project 2-2 — Simplified

This version follows the simple, sequential style of the GitHub notebook:
1. Set the study variables and import packages.
2. Search YouTube and randomly select the target videos.
3. Collect the requested data and create the CSV.
4. Add up to 10 preceding-video view counts for each target.
5. Try to retrieve transcripts and add them to the same CSV.

The sample is random within the larger set returned by YouTube Search. YouTube does not provide a globally random-search option.

## 1. Install, import, and set variables

In [2]:
%pip install -q google-api-python-client pandas isodate youtube-transcript-api

from datetime import datetime, timezone
import json
import random
import re

import isodate
import pandas as pd
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi
from my_api_info import youtube_api_key as apikey

youtube = build("youtube", "v3", developerKey=apikey)

# Change these variables to change the study
NICHE_QUERIES = ["fitness", "nutrition", "workout", "healthy eating", "weight loss"]
SAMPLE_SIZE = 500
PAST_VIDEO_COUNT = 10
RANDOM_SEED = 42
PUBLISHED_AFTER = "2025-09-01T00:00:00Z"
PUBLISHED_BEFORE = "2026-03-01T00:00:00Z"
VIRALITY_THRESHOLD = 2.5

OUTPUT_FILE = "virality_sample.csv"

Note: you may need to restart the kernel to use updated packages.


## 2. Pull a random sample of medium-length videos

YouTube defines `videoDuration="medium"` as 4–20 minutes. This cell collects a larger search pool and then randomly selects `SAMPLE_SIZE` IDs. Creator history does not affect selection.

In [7]:
candidate_ids = []
search_calls = 0
candidate_pool_size = min(SAMPLE_SIZE * 4, 5000)

for niche_query in NICHE_QUERIES:
    next_page_token = None

    while len(candidate_ids) < candidate_pool_size and search_calls < 100:
        search_parameters = {
            "part": "id",
            "q": niche_query,
            "type": "video",
            "videoDuration": "medium",
            "publishedAfter": PUBLISHED_AFTER,
            "publishedBefore": PUBLISHED_BEFORE,
            "maxResults": 50,
        }

        if next_page_token:
            search_parameters["pageToken"] = next_page_token

        response = youtube.search().list(**search_parameters).execute()
        search_calls += 1

        for item in response.get("items", []):
            video_id = item.get("id", {}).get("videoId")
            if video_id and video_id not in candidate_ids:
                candidate_ids.append(video_id)

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    if len(candidate_ids) >= candidate_pool_size or search_calls >= 100:
        break

if len(candidate_ids) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(candidate_ids)} videos were found. Expand the date range or NICHE_QUERIES."
    )

random.seed(RANDOM_SEED)
video_ids = random.sample(candidate_ids, SAMPLE_SIZE)

print("Search calls used:", search_calls)
print("Candidate videos found:", len(candidate_ids))
print("Random videos selected:", len(video_ids))

Search calls used: 38
Candidate videos found: 1445
Random videos selected: 500


## 3. Get video data and create the CSV

In [8]:
# Retrieve video data in batches of 50
videos = []

for start in range(0, len(video_ids), 50):
    batch = video_ids[start:start + 50]
    response = youtube.videos().list(
        part="snippet,statistics,contentDetails",
        id=",".join(batch),
    ).execute()

    for item in response.get("items", []):
        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})

        videos.append({
            "video_id": item["id"],
            "channel_id": snippet.get("channelId"),
            "title": snippet.get("title", ""),
            "description": snippet.get("description", ""),
            "channel": snippet.get("channelTitle", ""),
            "published_at": snippet.get("publishedAt"),
            "tags": snippet.get("tags", []),
            "views": int(statistics["viewCount"]) if statistics.get("viewCount") else None,
            "likes": int(statistics["likeCount"]) if statistics.get("likeCount") else None,
            "comments": int(statistics["commentCount"]) if statistics.get("commentCount") else None,
            "duration_minutes": isodate.parse_duration(
                item["contentDetails"]["duration"]
            ).total_seconds() / 60,
        })

# Retrieve subscriber counts and each channel's uploads playlist
channel_ids = list({video["channel_id"] for video in videos})
channel_data = {}

for start in range(0, len(channel_ids), 50):
    batch = channel_ids[start:start + 50]
    response = youtube.channels().list(
        part="statistics,contentDetails",
        id=",".join(batch),
    ).execute()

    for item in response.get("items", []):
        statistics = item.get("statistics", {})
        channel_data[item["id"]] = {
            "subscribers": None if statistics.get("hiddenSubscriberCount") else int(statistics.get("subscriberCount", 0)),
            "uploads_playlist": item.get("contentDetails", {}).get("relatedPlaylists", {}).get("uploads"),
        }

# Create the table and calculated columns
df = pd.DataFrame(videos)
df["subscribers"] = df["channel_id"].map(
    lambda channel_id: channel_data.get(channel_id, {}).get("subscribers")
)
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df["age_days"] = (
    pd.Timestamp.now(tz="UTC") - df["published_at"]
).dt.total_seconds() / 86400
df["tag_count"] = df["tags"].apply(len)
df["views_per_day"] = df["views"] / df["age_days"].clip(lower=1)
df["like_rate"] = df["likes"] / df["views"].replace(0, pd.NA)
df["comment_rate"] = df["comments"] / df["views"].replace(0, pd.NA)
df["views_per_subscriber"] = df["views"] / df["subscribers"].replace(0, pd.NA)
df["title_length"] = df["title"].str.len()
df["description_length"] = df["description"].str.len()
df["hashtag_count"] = df.apply(
    lambda row: len(re.findall(r"#[\w-]+", row["title"] + " " + row["description"])),
    axis=1,
)
df["tags"] = df["tags"].apply(lambda tags: json.dumps(tags, ensure_ascii=False))

output_columns = [
    "video_id", "title", "views", "likes", "comments",
    "duration_minutes", "tags", "tag_count", "published_at",
    "age_days", "channel", "subscribers", "views_per_day",
    "like_rate", "comment_rate", "views_per_subscriber",
    "title_length", "description_length", "hashtag_count",
]

df[output_columns].to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df)} videos to {OUTPUT_FILE}")
display(df[output_columns].head())

Saved 500 videos to virality_sample.csv


,video_id,title,views,likes,comments,duration_minutes,tags,tag_count,published_at,age_days,channel,subscribers,views_per_day,like_rate,comment_rate,views_per_subscriber,title_length,description_length,hashtag_count
0,zdXADzWzV_Y,31.3 Pounds Down | Big Tone's Weight Loss Update,7,0.0,1.0,11.933333,[],0,2026-01-16 03:32:05+00:00,306.854569,BigTone,5340,0.022812,0.0,0.142857,0.001311,48,294,5
1,-NQH5CcmZeI,"Female Bodybuilders Reveal Fitness Routines, N...",260303,3406.0,353.0,8.416667,[],0,2025-10-31 22:00:00+00:00,383.085182,Portia Show,40500,679.491174,0.013085,0.001356,6.427235,81,686,5
2,wFT0b81H4qc,Scientists Discovered the Exercise That Fights...,719180,11636.0,996.0,11.266667,[],0,2025-12-10 17:01:54+00:00,343.292196,FoundMyFitness Clips,559000,2094.950041,0.01618,0.001385,1.286547,78,929,0
3,Ag7Dui9Plys,The Most EFFECTIVE Total Body Workout (HIT EVE...,532375,14650.0,688.0,13.333333,"[""total body workout"", ""full body workout"", ""t...",24,2026-01-12 19:13:32+00:00,310.200784,ATHLEAN-X™,14300000,1716.227126,0.027518,0.001292,0.037229,56,4897,0
4,JNg7V6qL-8Y,SONBAHAR DEPRESYONU MU DEDİNİZ?,5882,498.0,0.0,4.900000,"[""depresyon nasıl geçer"", ""depresyonu azaltmak...",12,2025-09-17 17:00:26+00:00,427.293215,Prof. Dr. Muhammed Keskin,1280000,13.765723,0.084665,0.0,0.004595,31,2470,8


## 4. Add views from up to 10 preceding videos

This step never removes a target video. If fewer than 10 preceding public videos are available, it uses whatever is available.

In [11]:
from googleapiclient.errors import HttpError
def get_previous_video_ids(uploads_playlist, target_video_id):
    previous_ids = []
    target_found = False
    next_page_token = None
    pages_checked = 0

    while uploads_playlist and pages_checked < 20:
        request = {
            "part": "contentDetails",
            "playlistId": uploads_playlist,
            "maxResults": 50,
        }
        if next_page_token:
            request["pageToken"] = next_page_token

        try:
            response = youtube.playlistItems().list(**request).execute()
        except HttpError as error:
            # Some channels expose an uploads playlist ID that is no longer available.
            # Keep the target video and leave its baseline empty.
            if error.resp.status == 404:
                return []
            raise
        pages_checked += 1

        for item in response.get("items", []):
            video_id = item.get("contentDetails", {}).get("videoId")

            if video_id == target_video_id:
                target_found = True
            elif target_found and video_id:
                previous_ids.append(video_id)

            if len(previous_ids) == PAST_VIDEO_COUNT:
                return previous_ids

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return previous_ids

past_counts = []
past_ids = []
past_views = []
baseline_averages = []
virality_multipliers = []
viral_labels = []

for index, video in df.iterrows():
    uploads_playlist = channel_data.get(video["channel_id"], {}).get("uploads_playlist")
    previous_ids = get_previous_video_ids(uploads_playlist, video["video_id"])
    previous_views = []

    if previous_ids:
        response = youtube.videos().list(
            part="statistics",
            id=",".join(previous_ids),
        ).execute()
        previous_views = [
            int(item["statistics"]["viewCount"])
            for item in response.get("items", [])
            if item.get("statistics", {}).get("viewCount")
        ]

    baseline_average = (
        sum(previous_views) / len(previous_views)
        if previous_views else None
    )
    multiplier = (
        video["views"] / baseline_average
        if baseline_average and video["views"] is not None else None
    )

    past_counts.append(len(previous_views))
    past_ids.append(json.dumps(previous_ids))
    past_views.append(json.dumps(previous_views))
    baseline_averages.append(baseline_average)
    virality_multipliers.append(multiplier)
    viral_labels.append(None if multiplier is None else multiplier >= VIRALITY_THRESHOLD)

    if (index + 1) % 25 == 0:
        print(f"Baselines completed: {index + 1}/{len(df)}")

df["past_video_count_used"] = past_counts
df["past_video_ids"] = past_ids
df["past_video_views"] = past_views
df["baseline_average_views"] = baseline_averages
df["virality_multiplier"] = virality_multipliers
df["is_viral_2_5x"] = viral_labels

baseline_columns = [
    "past_video_count_used", "past_video_ids", "past_video_views",
    "baseline_average_views", "virality_multiplier", "is_viral_2_5x",
]
df[output_columns + baseline_columns].to_csv(OUTPUT_FILE, index=False)
print(f"Added preceding-video baselines to {OUTPUT_FILE}")
display(df[["video_id"] + baseline_columns].head())

Baselines completed: 25/500
Baselines completed: 50/500
Baselines completed: 75/500
Baselines completed: 100/500
Baselines completed: 125/500
Baselines completed: 150/500
Baselines completed: 175/500
Baselines completed: 200/500
Baselines completed: 225/500
Baselines completed: 250/500
Baselines completed: 275/500
Baselines completed: 300/500
Baselines completed: 325/500
Baselines completed: 350/500
Baselines completed: 375/500
Baselines completed: 400/500
Baselines completed: 425/500
Baselines completed: 450/500
Baselines completed: 475/500
Baselines completed: 500/500
Added preceding-video baselines to virality_sample.csv


,video_id,past_video_count_used,past_video_ids,past_video_views,baseline_average_views,virality_multiplier,is_viral_2_5x
0,zdXADzWzV_Y,10,"[""MfnVsYiAoLk"", ""VRPsB6cBB6A"", ""f42Z4cOP-T8"", ...","[212, 18, 135, 9, 600, 14, 139, 15, 76799, 91]",7803.2,0.000897,False
1,-NQH5CcmZeI,10,"[""-TlUK5NibUA"", ""nWUtTgvPhlg"", ""bXZKslLW4yk"", ...","[1169, 37707, 415, 1226, 5916, 1172, 3366, 467...",5385.6,48.333148,True
2,wFT0b81H4qc,10,"[""fPmJ8N6Q350"", ""LB8TKaspMik"", ""1mou2TUOC1k"", ...","[17993, 86100, 47498, 8175, 9690, 114829, 1817...",51411.4,13.988726,True
3,Ag7Dui9Plys,10,"[""SPC37KpFpTU"", ""gLz05PhtlPg"", ""-iV1hh4vPEc"", ...","[626156, 271039, 362867, 239190, 721791, 41651...",1201198.3,0.443203,False
4,JNg7V6qL-8Y,10,"[""TVqf9LbPO2k"", ""hvXGp4k3ZSI"", ""1R9Or_yNKHc"", ...","[282103, 234164, 320004, 207930, 225713, 29491...",327024.4,0.017986,False


## 5. Try transcripts and add them to the CSV

Transcript retrieval is separate from the official YouTube Data API. Videos remain in the dataset when captions are missing or retrieval fails.

In [17]:
import time
import pandas as pd
from youtube_transcript_api import YouTubeTranscriptApi

OUTPUT_FILE = "virality_sample.csv"
DELAY_SECONDS = 10  # Pause between transcript requests

df = pd.read_csv(OUTPUT_FILE)
ytt_api = YouTubeTranscriptApi()

# Create transcript columns on the first run. On later runs, keep successes
# and retry blocked requests plus videos that had no English transcript.
if "transcript" not in df.columns:
    df["transcript"] = pd.NA
if "transcript_word_count" not in df.columns:
    df["transcript_word_count"] = 0
if "transcript_status" not in df.columns:
    df["transcript_status"] = pd.NA
if "transcript_language" not in df.columns:
    df["transcript_language"] = pd.NA

all_rows_to_try = df.index[
    df["transcript_status"].isna()
    | df["transcript_status"].eq("IpBlocked")
    | df["transcript_status"].eq("RequestBlocked")
    | df["transcript_status"].eq("NoTranscriptFound")
].tolist()
rows_to_try = all_rows_to_try

print(f"Transcript rows remaining: {len(all_rows_to_try)}")
print(f"Will attempt all {len(rows_to_try)} remaining rows unless an IP block occurs.")

for attempt_number, row_index in enumerate(rows_to_try, start=1):
    video_id = df.at[row_index, "video_id"]

    try:
        available_transcripts = list(ytt_api.list(video_id))
        english_transcripts = [
            item for item in available_transcripts
            if item.language_code.lower().startswith("en")
        ]
        selected_transcript = (
            english_transcripts[0] if english_transcripts else available_transcripts[0]
        )
        transcript = selected_transcript.fetch()
        full_transcript = " ".join(segment.text for segment in transcript)
        df.at[row_index, "transcript"] = full_transcript
        df.at[row_index, "transcript_word_count"] = len(full_transcript.split())
        df.at[row_index, "transcript_status"] = "success"
        df.at[row_index, "transcript_language"] = selected_transcript.language_code
    except Exception as error:
        error_name = type(error).__name__
        df.at[row_index, "transcript"] = None
        df.at[row_index, "transcript_word_count"] = 0
        df.at[row_index, "transcript_status"] = error_name
        df.at[row_index, "transcript_language"] = pd.NA

    # Save each result immediately so the CSV acts as a checkpoint.
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"Transcript attempts completed: {attempt_number}/{len(rows_to_try)}")

    # Stop instead of repeatedly sending requests from an already-blocked IP.
    if df.at[row_index, "transcript_status"] in ["IpBlocked", "RequestBlocked"]:
        print("ip block error")
        break

    time.sleep(DELAY_SECONDS)

df.to_csv(OUTPUT_FILE, index=False)

transcript_columns = [
    "transcript", "transcript_word_count", "transcript_status", "transcript_language"
]
print("Successful transcripts:", (df["transcript_status"] == "success").sum())
print(f"Final dataset saved to {OUTPUT_FILE}")
display(df[["video_id"] + transcript_columns].head())

Transcript rows remaining: 292
Will attempt all 292 remaining rows unless an IP block occurs.
Transcript attempts completed: 1/292
ip block error
Successful transcripts: 183
Final dataset saved to virality_sample.csv


,video_id,transcript,transcript_word_count,transcript_status,transcript_language
0,zdXADzWzV_Y,"What's up, guys? It's Big Tone here. Today, um...",1418,success,NaN
1,-NQH5CcmZeI,[Music] Hello and welcome to the show. You kno...,1797,success,NaN
2,wFT0b81H4qc,"Okay. So, let's I think that's pretty clear th...",2155,success,NaN
3,Ag7Dui9Plys,If you're looking for a full body workout that...,2739,success,NaN
4,JNg7V6qL-8Y,Sonbahar mevsimine girdiğimiz bugünlerde istek...,658,success,tr
